# Bottleneck study — constraints ∧ WAPE ∧ demand response (hist, Colab)

Runtime > Change runtime type > **GPU** (any). The repo ships the prepared data
(`data/preprocessed/hist/...`) and every pilot checkpoint (`weights/`), so cloning is
the whole setup. Scripts re-run from cached JSONs — a disconnect costs one cell.

**The three bottlenecks** (why this study exists):

1. model WAPE loses to 5-min persistence (0.1084)
2. the constrained models default to persistence and ignore a simulated load increase
   (RAYEN co-predicts its own demand `D_t`, so the balance plane never moves)
3. the battery/SOC constraint reads inconsistently (79% vs 115% vs 4220% "swing"
   for the same physics, depending only on the integration window)

## 1 · Metrics & evaluation

| what | metric | protocol |
| --- | --- | --- |
| accuracy | WAPE per-channel + macro, R² | teacher-forced full test (Jan–Jul 2026) |
| balance | `bal_own_max_mw`, mismatch % | teacher-forced + scenario rollout |
| ramp / box | `n_ramp` (free/seam/tf split), `n_neg` | closed-loop: stress episodes + free-window rollout |
| responsiveness | **response_capture** (fraction of simulated increase delivered), tracking p50/p95 | free-window closed-loop scenario rollout |
| battery | per-day SOC feasible %, worst-day % (whole-window swing reported for transparency only) | all protocols, unified |

Baselines every row must face: **persistence** (WAPE floor) and **anchor(persistence)**
(zero-parameter, structurally responsive).

## 2 · Solution ladders (A → B → C)

**P2 — respond to the simulated load increase** *(headline)*
A: pin `D = nd(t−1)` (`RayenHeadFixedD`) — the scenario `net_demand` is teacher-forced
into the free window, so the plane moves with the simulated demand and the fleet is
*forced* to follow within ramps. → B: retrain `*_rayenfd` (gas_steam passthrough on).
→ C: inference-only anchor (`anchor_persistence` row, always reported).

**P1 — WAPE parity with persistence**
A: freeze gas_steam at persistence (`passthrough`, zero-retrain buffer retrofit).
→ B: retrain. → C: accept macro gap, isolate it to the battery channels per-channel.

**P3 — battery constraint**
A: per-day SOC segmentation everywhere (batteries cycle daily; window-cumsum drift
was the artifact). → B: stateful closed-loop SOC clip. → C: recalibrate cap+η from
the telemetry subset.

Gates are computed by `constraints/study_report.py` → `study_verdicts.json`; the
conditional cells below read it.


In [ ]:
REPO = "github.com/nm-quan/energy_modelling.git"
TOKEN = ""   # repo is PRIVATE: paste a fine-grained READ-ONLY token (Contents: read)
BRANCH = "claude/model-bottlenecks-constraints-gb1aoj"   # set to "main" once merged
import os
url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
if not os.path.exists("energy_modelling"):
    !git clone -q --branch $BRANCH $url
%cd energy_modelling
!git pull -q
!nvidia-smi -L


In [ ]:
# optional: mount Drive so retrained weights survive the VM
OUT = "/content/runs"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/energy_runs"
except Exception as e:
    print("no Drive:", e)
os.makedirs(OUT, exist_ok=True)


## 0 · Sanity — constraint layers are feasible at any weights


In [ ]:
!python constraints/test_constraint_layers.py


## 1 · Pilot — ladder **A** for all three problems, no training

`eval_hist_models.py` runs the TF (balance/accuracy) + CL (stress-episode ramp)
tables over every checkpoint in `weights/`, now including `itransformer_rayenfd`
with the gas_steam passthrough retrofit (`+spt`, P1-A) and per-channel WAPE +
per-day SOC (P3-A) built in.


In [ ]:
!python constraints/eval_hist_models.py --ckpt-dir weights
from IPython.display import Markdown, display
for t in ["hist_tf.md", "hist_tf_per_channel.md", "hist_cl_stress.md"]:
    display(Markdown(open(f"constraints/results/{t}").read()))


In [ ]:
# P2-A: scenario rollouts. reshape = legacy-comparable energy-neutral shift;
# increase g=10 = the simulated increased-load period (headline gate).
# Rollouts are sequential (batch 2) — a few min per model; GPU helps little.
!python demand_simulation/study_shift.py --scenario reshape
!python demand_simulation/study_shift.py --scenario increase --g 10
import glob
for t in sorted(glob.glob("demand_simulation/sweep_eqnd/study/*.md")):
    display(Markdown(open(t).read()))


## 2 · Gates — decide the ladders


In [ ]:
!python constraints/study_report.py
import json
V = json.load(open("constraints/results/study_verdicts.json"))
RUN_RETRAIN = V["run_retrain"]
for k in ("p1_accuracy", "p2_responsiveness", "p3_battery"):
    print(f"{k:20s} pass={V[k]['pass']}  {V[k]['why']}")
print("\nRUN_RETRAIN =", RUN_RETRAIN)


## 3 · Ladder **B** (conditional) — retrain `rayenfd`, passthrough on

Only runs if a gate failed. `make_rayen` now defaults `passthrough_idx=(2,)`, so a
fresh train bakes the gas_steam freeze in (vs the zero-retrain buffer retrofit the
pilot used). ~40 epochs; checkpoints per epoch and auto-resumes — re-run on disconnect.


In [ ]:
if RUN_RETRAIN:
    for arch in ("itransformer_rayenfd", "lstm_rayenfd"):
        !python ml/train_hist.py --arch $arch --seed 0 --out $OUT
        !cp $OUT/{arch}_hist_s0.pt weights/
    # re-run the pilot on the new checkpoints, then re-gate
    !python constraints/eval_hist_models.py --ckpt-dir weights
    !python demand_simulation/study_shift.py --scenario increase --g 10
    !python constraints/study_report.py
    V = json.load(open("constraints/results/study_verdicts.json"))
    for k in ("p1_accuracy", "p2_responsiveness", "p3_battery"):
        print(f"{k:20s} pass={V[k]['pass']}  {V[k]['why']}")
else:
    print("all gates passed — A suffices, no retrain (C fallback = anchor_persistence row)")


## 4 · Response curve — how much simulated load increase can the fleet follow?

Sweep the increase magnitude g. The demand-cap guard (10,783.7 MW historical max)
skips any g whose reshaped demand would exceed it; beyond the fleet's one-step ramp
capacity the tracking residual grows — that residual is the *physical* ceiling, and
`ramp` must stay 0 (balance yields to ramps, never the reverse).


In [ ]:
import subprocess
for g in (5, 20, 30):   # 10 already ran in the pilot
    r = subprocess.run(["python", "demand_simulation/study_shift.py",
                        "--scenario", "increase", "--g", str(g)])
    if r.returncode:
        print(f"g={g}% skipped (demand cap or error above)")


## 5 · Final report — verdicts, per-channel table, response curve, trade-off figure


In [ ]:
!python constraints/study_report.py
display(Markdown(open("constraints/results/study_summary.md").read()))
from IPython.display import Image
Image("constraints/results/figure/study_tradeoff.png")


In [ ]:
# keep the numbers: zip results (+ any retrained weights already sync to Drive via $OUT)
!zip -qr study_results.zip constraints/results demand_simulation/sweep_eqnd/study weights/*rayenfd*
try:
    from google.colab import files
    files.download("study_results.zip")
except Exception:
    print("study_results.zip written")
